In [86]:
from ultralytics import YOLO
import cv2
import numpy as np
import cv2

In [87]:
kalman = cv2.KalmanFilter(4,2)

kalman.measurementMatrix = np.array([
    [1,0,0,0],
    [0,1,0,0]
], np.float32)

kalman.transitionMatrix = np.array([
    [1,0,1,0],
    [0,1,0,1],
    [0,0,1,0],
    [0,0,0,1]
], np.float32)

kalman.processNoiseCov = np.eye(4, dtype=np.float32) * 0.03

In [89]:
model = YOLO("../notebooks/runs/detect/train/weights/best.pt")

cap = cv2.VideoCapture("../videos/cricket_ball_demo.mp4")

width = int(cap.get(3))
height = int(cap.get(4))
fps = int(cap.get(5))

out = cv2.VideoWriter(
    "../videos/trajectory_output.mp4",
    cv2.VideoWriter_fourcc(*'mp4v'),
    fps,
    (width, height)
)

trajectory = []
smooth_trajectory = []
frame_count = 0
MAX_TRAIL = 400

while True:
    
    ret, frame = cap.read()
    if not ret:
        break

    results = model.track(frame, conf=0.4, persist=True)[0]

    if results.boxes is not None:

        boxes = results.boxes.xyxy.cpu().numpy()

        if len(boxes) > 0:

            x1, y1, x2, y2 = boxes[0]

            cx = int((x1+x2)/2)
            cy = int((y1+y2)/2)

            measurement = np.array([[np.float32(cx)],
                                    [np.float32(cy)]])

            kalman.correct(measurement)
            prediction = kalman.predict()

            cx = int(prediction[0][0])
            cy = int(prediction[1][0])

            FRAME_SKIP = 3
            MIN_MOVE = 6

            if len(trajectory) == 0:
                trajectory.append((cx,cy))

            elif frame_count % FRAME_SKIP == 0:

                prev = trajectory[-1]

                dist = np.sqrt((cx-prev[0])**2 + (cy-prev[1])**2)

                if dist > MIN_MOVE:
                    trajectory.append((cx,cy))

            WINDOW = 12

            if len(trajectory) >= WINDOW:

                avg_x = int(np.mean([p[0] for p in trajectory[-WINDOW:]]))
                avg_y = int(np.mean([p[1] for p in trajectory[-WINDOW:]]))

                smooth_trajectory.append((avg_x, avg_y))

            cv2.rectangle(
                frame,
                (int(x1),int(y1)),
                (int(x2),int(y2)),
                (0,255,0),
                2
            )

    # draw trajectory
    for i in range(1, min(len(smooth_trajectory), MAX_TRAIL)):
        cv2.line(frame,
             smooth_trajectory[-i],
             smooth_trajectory[-i-1],
             (0,0,255),
             2)

    out.write(frame)

    frame_count +=1 

cap.release()
out.release()


0: 640x384 (no detections), 59.1ms
Speed: 7.2ms preprocess, 59.1ms inference, 10.7ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 (no detections), 43.7ms
Speed: 1.1ms preprocess, 43.7ms inference, 0.2ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 (no detections), 40.7ms
Speed: 0.9ms preprocess, 40.7ms inference, 0.3ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 (no detections), 42.4ms
Speed: 1.1ms preprocess, 42.4ms inference, 0.5ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 (no detections), 43.7ms
Speed: 1.0ms preprocess, 43.7ms inference, 0.2ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 (no detections), 40.9ms
Speed: 1.1ms preprocess, 40.9ms inference, 0.2ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 (no detections), 40.6ms
Speed: 0.9ms preprocess, 40.6ms inference, 0.2ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 (no detections), 41.7ms
Speed: 1.1ms preprocess, 41.7ms 